In [3]:
import pandas as pd
import numpy as np
from datasets import load_dataset

In [5]:
dataset = load_dataset("ucberkeley-dlab/measuring-hate-speech")
dataset

data/train-00000-of-00001.parquet:   0%|          | 0.00/20.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/135556 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['comment_id', 'annotator_id', 'platform', 'sentiment', 'respect', 'insult', 'humiliate', 'status', 'dehumanize', 'violence', 'genocide', 'attack_defend', 'hatespeech', 'hate_speech_score', 'text', 'infitms', 'outfitms', 'annotator_severity', 'std_err', 'annotator_infitms', 'annotator_outfitms', 'hypothesis', 'target_race_asian', 'target_race_black', 'target_race_latinx', 'target_race_middle_eastern', 'target_race_native_american', 'target_race_pacific_islander', 'target_race_white', 'target_race_other', 'target_race', 'target_religion_atheist', 'target_religion_buddhist', 'target_religion_christian', 'target_religion_hindu', 'target_religion_jewish', 'target_religion_mormon', 'target_religion_muslim', 'target_religion_other', 'target_religion', 'target_origin_immigrant', 'target_origin_migrant_worker', 'target_origin_specific_country', 'target_origin_undocumented', 'target_origin_other', 'target_origin', 'target_gender_men', 'target

In [6]:
df = dataset["train"].to_pandas()

print("Shape:", df.shape)
df.head()

Shape: (135556, 143)


,comment_id,annotator_id,platform,sentiment,respect,insult,humiliate,status,dehumanize,violence,...,annotator_religion_hindu,annotator_religion_jewish,annotator_religion_mormon,annotator_religion_muslim,annotator_religion_nothing,annotator_religion_other,annotator_sexuality_bisexual,annotator_sexuality_gay,annotator_sexuality_straight,annotator_sexuality_other
0,47777,10873,3,0.0,0.0,0.0,0.0,2.0,0.0,0.0,...,False,False,False,False,False,False,False,False,True,False
1,39773,2790,2,0.0,0.0,0.0,0.0,2.0,0.0,0.0,...,False,False,False,False,False,False,False,False,True,False
2,47101,3379,3,4.0,4.0,4.0,4.0,4.0,4.0,0.0,...,False,False,False,False,True,False,False,False,True,False
3,43625,7365,3,2.0,3.0,2.0,1.0,2.0,0.0,0.0,...,False,False,False,False,False,False,False,False,True,False
4,12538,488,0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,False,False,False,False,False,False,False,False,True,False


In [7]:
for i, col in enumerate(df.columns):
    print(i, col)

0 comment_id
1 annotator_id
2 platform
3 sentiment
4 respect
5 insult
6 humiliate
7 status
8 dehumanize
9 violence
10 genocide
11 attack_defend
12 hatespeech
13 hate_speech_score
14 text
15 infitms
16 outfitms
17 annotator_severity
18 std_err
19 annotator_infitms
20 annotator_outfitms
21 hypothesis
22 target_race_asian
23 target_race_black
24 target_race_latinx
25 target_race_middle_eastern
26 target_race_native_american
27 target_race_pacific_islander
28 target_race_white
29 target_race_other
30 target_race
31 target_religion_atheist
32 target_religion_buddhist
33 target_religion_christian
34 target_religion_hindu
35 target_religion_jewish
36 target_religion_mormon
37 target_religion_muslim
38 target_religion_other
39 target_religion
40 target_origin_immigrant
41 target_origin_migrant_worker
42 target_origin_specific_country
43 target_origin_undocumented
44 target_origin_other
45 target_origin
46 target_gender_men
47 target_gender_non_binary
48 target_gender_transgender_men
49 target_

In [8]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 135556
Columns: 143


In [10]:
missing = df.isnull().sum().sort_values(ascending=False)
missing.head(30)

annotator_age                  105
annotator_income               103
annotator_ideology              27
annotator_educ                  17
comment_id                       0
annotator_id                     0
platform                         0
status                           0
dehumanize                       0
violence                         0
genocide                         0
sentiment                        0
respect                          0
insult                           0
humiliate                        0
text                             0
hate_speech_score                0
hatespeech                       0
attack_defend                    0
infitms                          0
outfitms                         0
annotator_severity               0
std_err                          0
target_race_black                0
target_race_latinx               0
target_race_middle_eastern       0
target_race_native_american      0
annotator_infitms                0
annotator_outfitms  

In [11]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate comment IDs:", df["comment_id"].duplicated().sum())

Duplicate rows: 0
Duplicate comment IDs: 95991


In [12]:
df["annotator_gender"].value_counts(dropna=False)

annotator_gender
female               76370
male                 57582
non-binary             985
prefer_not_to_say      500
self-describe          119
Name: count, dtype: int64

In [14]:
df["annotator_ideology"].value_counts(dropna=False)

annotator_ideology
liberal                   33812
neutral                   23112
slightly_liberal          21333
extremely_liberal         17944
conservative              15628
slightly_conservative     15101
extremely_conservative     4544
no_opinion                 4055
NaN                          27
Name: count, dtype: int64

In [15]:
df.to_csv("../data/raw/mhs_raw.csv", index=False)
print("Saved raw dataset to ../data/raw/mhs_raw.csv")

Saved raw dataset to ../data/raw/mhs_raw.csv


In [16]:
schema_path = "../results/tables/mhs_schema_audit.csv"

schema_df = pd.DataFrame({
    "column_name": df.columns,
    "dtype": [str(df[col].dtype) for col in df.columns],
    "missing_count": [df[col].isnull().sum() for col in df.columns],
    "missing_percent": [round(df[col].isnull().mean() * 100, 2) for col in df.columns],
    "unique_values": [df[col].nunique() for col in df.columns]
})

schema_df.to_csv(schema_path, index=False)

print(f"Saved schema audit to {schema_path}")
schema_df

Saved schema audit to ../results/tables/mhs_schema_audit.csv


,column_name,dtype,missing_count,missing_percent,unique_values
0,comment_id,int32,0,0.0,39565
1,annotator_id,int32,0,0.0,7912
2,platform,int8,0,0.0,4
3,sentiment,float64,0,0.0,5
4,respect,float64,0,0.0,5
...,...,...,...,...,...
138,annotator_religion_other,bool,0,0.0,2
139,annotator_sexuality_bisexual,bool,0,0.0,2
140,annotator_sexuality_gay,bool,0,0.0,2
141,annotator_sexuality_straight,bool,0,0.0,2


In [18]:
survey_item_cols = [
    "sentiment",
    "respect",
    "insult",
    "humiliate",
    "status",
    "dehumanize",
    "violence",
    "genocide",
    "attack_defend",
    "hatespeech"
]

df[survey_item_cols].head()

,sentiment,respect,insult,humiliate,status,dehumanize,violence,genocide,attack_defend,hatespeech
0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,2.0,0.0
2,4.0,4.0,4.0,4.0,4.0,4.0,0.0,0.0,4.0,2.0
3,2.0,3.0,2.0,1.0,2.0,0.0,0.0,0.0,3.0,0.0
4,4.0,4.0,4.0,4.0,4.0,4.0,4.0,1.0,3.0,2.0


In [20]:
for col in survey_item_cols:
    print(col, "->", col in df.columns)

sentiment -> True
respect -> True
insult -> True
humiliate -> True
status -> True
dehumanize -> True
violence -> True
genocide -> True
attack_defend -> True
hatespeech -> True


In [19]:
for col in survey_item_cols:
    print("\n" + "="*50)
    print(col)
    print(df[col].value_counts(dropna=False).sort_index())


sentiment
sentiment
0.0     9658
1.0     9435
2.0    19785
3.0    35243
4.0    61435
Name: count, dtype: int64

respect
respect
0.0    12839
1.0     9629
2.0    23895
3.0    30720
4.0    58473
Name: count, dtype: int64

insult
insult
0.0    18476
1.0    14667
2.0    18587
3.0    39673
4.0    44153
Name: count, dtype: int64

humiliate
humiliate
0.0    21306
1.0    19555
2.0    25035
3.0    39382
4.0    30278
Name: count, dtype: int64

status
status
0.0     1228
1.0     4261
2.0    61154
3.0    36413
4.0    32500
Name: count, dtype: int64

dehumanize
dehumanize
0.0    31855
1.0    28434
2.0    25292
3.0    28653
4.0    21322
Name: count, dtype: int64

violence
violence
0.0    67922
1.0    30727
2.0    12241
3.0    11262
4.0    13404
Name: count, dtype: int64

genocide
genocide
0.0    90058
1.0    22838
2.0     8107
3.0     5301
4.0     9252
Name: count, dtype: int64

attack_defend
attack_defend
0.0     7958
1.0    11046
2.0    38201
3.0    44883
4.0    33468
Name: count, dtype: int64

h

In [27]:
important_cols = [
    "comment_id",
    "annotator_id",
    "text",
    "annotator_gender",
    "annotator_ideology"
] + survey_item_cols

df[important_cols].head()



,comment_id,annotator_id,text,annotator_gender,annotator_ideology,sentiment,respect,insult,humiliate,status,dehumanize,violence,genocide,attack_defend,hatespeech
0,47777,10873,Yes indeed. She sort of reminds me of the elde...,male,neutral,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0
1,39773,2790,The trans women reading this tweet right now i...,female,neutral,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,2.0,0.0
2,47101,3379,Question: These 4 broads who criticize America...,male,slightly_conservative,4.0,4.0,4.0,4.0,4.0,4.0,0.0,0.0,4.0,2.0
3,43625,7365,It is about time for all illegals to go back t...,male,neutral,2.0,3.0,2.0,1.0,2.0,0.0,0.0,0.0,3.0,0.0
4,12538,488,For starters bend over the one in pink and kic...,female,neutral,4.0,4.0,4.0,4.0,4.0,4.0,4.0,1.0,3.0,2.0


In [29]:
survey_audit = []

for col in survey_item_cols:
    survey_audit.append({
        "column": col,
        "missing_count": df[col].isnull().sum(),
        "missing_percent": round(df[col].isnull().mean() * 100, 2),
        "unique_values": df[col].nunique(),
        "min": df[col].min(),
        "max": df[col].max()
    })

survey_audit_df = pd.DataFrame(survey_audit)
survey_audit_df.to_csv("../results/tables/survey_item_audit.csv", index=False)

survey_audit_df

,column,missing_count,missing_percent,unique_values,min,max
0,sentiment,0,0.0,5,0.0,4.0
1,respect,0,0.0,5,0.0,4.0
2,insult,0,0.0,5,0.0,4.0
3,humiliate,0,0.0,5,0.0,4.0
4,status,0,0.0,5,0.0,4.0
5,dehumanize,0,0.0,5,0.0,4.0
6,violence,0,0.0,5,0.0,4.0
7,genocide,0,0.0,5,0.0,4.0
8,attack_defend,0,0.0,5,0.0,4.0
9,hatespeech,0,0.0,3,0.0,2.0


In [30]:
for col in survey_item_cols:
    print("\n" + "="*60)
    print(col)
    print(df[col].value_counts(dropna=False).sort_index())


sentiment
sentiment
0.0     9658
1.0     9435
2.0    19785
3.0    35243
4.0    61435
Name: count, dtype: int64

respect
respect
0.0    12839
1.0     9629
2.0    23895
3.0    30720
4.0    58473
Name: count, dtype: int64

insult
insult
0.0    18476
1.0    14667
2.0    18587
3.0    39673
4.0    44153
Name: count, dtype: int64

humiliate
humiliate
0.0    21306
1.0    19555
2.0    25035
3.0    39382
4.0    30278
Name: count, dtype: int64

status
status
0.0     1228
1.0     4261
2.0    61154
3.0    36413
4.0    32500
Name: count, dtype: int64

dehumanize
dehumanize
0.0    31855
1.0    28434
2.0    25292
3.0    28653
4.0    21322
Name: count, dtype: int64

violence
violence
0.0    67922
1.0    30727
2.0    12241
3.0    11262
4.0    13404
Name: count, dtype: int64

genocide
genocide
0.0    90058
1.0    22838
2.0     8107
3.0     5301
4.0     9252
Name: count, dtype: int64

attack_defend
attack_defend
0.0     7958
1.0    11046
2.0    38201
3.0    44883
4.0    33468
Name: count, dtype: int64

h

In [31]:
summary_path = "../results/tables/mhs_dataset_summary.txt"

with open(summary_path, "w", encoding="utf-8") as f:

    f.write("MEASURING HATE SPEECH (MHS) DATASET SUMMARY\n")
    f.write("=" * 60 + "\n\n")

    # Basic shape
    f.write(f"Total Rows: {df.shape[0]}\n")
    f.write(f"Total Columns: {df.shape[1]}\n\n")

    # Core counts
    f.write(f"Unique Comments: {df['comment_id'].nunique()}\n")
    f.write(f"Unique Annotators: {df['annotator_id'].nunique()}\n")
    f.write(f"Duplicate Full Rows: {df.duplicated().sum()}\n\n")

    # Survey items
    f.write("Survey Item Columns:\n")
    for col in survey_item_cols:
        f.write(f"- {col}\n")

    f.write("\n")

    # Annotator demographics
    f.write("Annotator Gender Distribution:\n")
    f.write(str(df["annotator_gender"].value_counts(dropna=False)))
    f.write("\n\n")

    f.write("Annotator Political Ideology Distribution:\n")
    f.write(str(df["annotator_ideology"].value_counts(dropna=False)))
    f.write("\n\n")

    # Missing values
    f.write("Top 20 Columns with Missing Values:\n")
    missing = df.isnull().sum().sort_values(ascending=False).head(20)
    f.write(str(missing))
    f.write("\n\n")

    # Survey item ranges
    f.write("Survey Item Statistics:\n")

    for col in survey_item_cols:
        f.write(f"\n{col}\n")
        f.write(str(df[col].describe()))
        f.write("\n")

print(f"Summary saved to: {summary_path}")

Summary saved to: ../results/tables/mhs_dataset_summary.txt


In [32]:
for col in df.columns:
    print(col)

comment_id
annotator_id
platform
sentiment
respect
insult
humiliate
status
dehumanize
violence
genocide
attack_defend
hatespeech
hate_speech_score
text
infitms
outfitms
annotator_severity
std_err
annotator_infitms
annotator_outfitms
hypothesis
target_race_asian
target_race_black
target_race_latinx
target_race_middle_eastern
target_race_native_american
target_race_pacific_islander
target_race_white
target_race_other
target_race
target_religion_atheist
target_religion_buddhist
target_religion_christian
target_religion_hindu
target_religion_jewish
target_religion_mormon
target_religion_muslim
target_religion_other
target_religion
target_origin_immigrant
target_origin_migrant_worker
target_origin_specific_country
target_origin_undocumented
target_origin_other
target_origin
target_gender_men
target_gender_non_binary
target_gender_transgender_men
target_gender_transgender_unspecified
target_gender_transgender_women
target_gender_women
target_gender_other
target_gender
target_sexuality_bisexu